# Phase 0 - Bag生成 + 弱标签标注（脉冲信号）

任务：
- 读取3个脉冲CSV
- 对每个Ori_Recording_XX.wav下采样到48kHz
- 切割成60秒非重叠bag
- 根据Burst/Buzz/Click在bag内是否有重叠，打标签1/0
- 统计每个.wav文件的positive/negative bag数量

In [12]:
import os
import pandas as pd
import torchaudio
import torch
import numpy as np
from tqdm import tqdm

# ====================== 配置 ======================
root = r'D:\Project_Github\audio_click_mil'
original_audio_dir = os.path.join(root, 'data', 'original_audio')

# CSV路径
csv_paths = {
    'burst': os.path.join(root, 'data', 'BurstPulseTrains.csv'),
    'buzz': os.path.join(root, 'data', 'BuzzTrains.csv'),
    'click': os.path.join(root, 'data', 'ClickTrains.csv')
}

fs_target = 48000          # 下采样目标
bag_duration = 20.0        # 秒
print('配置完成')

配置完成


In [13]:
# 读取三个CSV并合并pulse事件
pulse_df = []
for name, path in csv_paths.items():
    df = pd.read_csv(path)
    df['type'] = name
    df = df[['Ori_file_num(No.)', 'Train_start(s)', 'Train_end(s)', 'type']].copy()
    df.rename(columns={'Ori_file_num(No.)': 'file_num'}, inplace=True)
    pulse_df.append(df)

all_pulses = pd.concat(pulse_df, ignore_index=True)
print(f'总脉冲事件数量: {len(all_pulses)}')
print(all_pulses.head())

总脉冲事件数量: 897
   file_num  Train_start(s)  Train_end(s)   type
0         1       1631.0973     1631.1532  burst
1         5        190.8192      191.8563  burst
2         6       1122.9183     1123.0090  burst
3         6       1276.3410     1276.7285  burst
4         6       1487.7401     1488.1118  burst


In [14]:
# 获取所有wav文件
wav_files = [f for f in os.listdir(original_audio_dir) if f.endswith('.wav') and f.startswith('Ori_Recording_')]
wav_files.sort(key=lambda x: int(x[-6:-4]))  # 按01,02排序

stats = []

for wav_name in tqdm(wav_files):
    file_num = int(wav_name[-6:-4])                     # Ori_Recording_01 → 1
    wav_path = os.path.join(original_audio_dir, wav_name)
    
    # 加载并下采样
    try:
        waveform, sr = torchaudio.load(wav_path, backend="soundfile")
    except:
        # 兜底方案：用 soundfile + torchaudio
        import soundfile as sf
        data, sr = sf.read(wav_path)
        waveform = torch.from_numpy(data).unsqueeze(0) if len(data.shape) == 1 else torch.from_numpy(data).T
        waveform = waveform.float()

    if sr != fs_target:
        resampler = torchaudio.transforms.Resample(sr, fs_target)
        waveform = resampler(waveform)
    
    total_seconds = waveform.shape[1] / fs_target
    
    # 计算bag数量
    num_bags = int(total_seconds // bag_duration)
    
    # 获取该文件的pulse事件
    file_pulses = all_pulses[all_pulses['file_num'] == file_num]
    
    positive_count = 0
    for i in range(num_bags):
        bag_start = i * bag_duration
        bag_end = bag_start + bag_duration
        
        # 判断是否有任何pulse与当前bag重叠
        overlap = file_pulses[(file_pulses['Train_start(s)'] < bag_end) & 
                             (file_pulses['Train_end(s)'] > bag_start)]
        if len(overlap) > 0:
            positive_count += 1
    
    negative_count = num_bags - positive_count
    
    stats.append({
        'file': wav_name,
        'file_num': file_num,
        'total_seconds': round(total_seconds, 2),
        'num_bags': num_bags,
        'positive_bags': positive_count,
        'negative_bags': negative_count,
        'positive_ratio': round(positive_count/num_bags, 4) if num_bags > 0 else 0
    })

# 保存统计结果
stats_df = pd.DataFrame(stats)
stats_df.to_csv(os.path.join(root, 'results', 'phase0_bag_stats.csv'), index=False)
print('统计完成！')
print(stats_df)

100%|██████████| 35/35 [05:00<00:00,  8.58s/it]

统计完成！
                    file  file_num  total_seconds  num_bags  positive_bags  \
0   Ori_Recording_01.wav         1        1799.44        89              9   
1   Ori_Recording_02.wav         2        1799.95        89             41   
2   Ori_Recording_03.wav         3        1799.97        89              6   
3   Ori_Recording_04.wav         4        1799.93        89              8   
4   Ori_Recording_05.wav         5        1799.95        89             12   
5   Ori_Recording_06.wav         6        1799.86        89             29   
6   Ori_Recording_07.wav         7        1799.94        89              7   
7   Ori_Recording_08.wav         8        1799.95        89             21   
8   Ori_Recording_09.wav         9        1799.28        89             30   
9   Ori_Recording_10.wav        10        1748.68        87             37   
10  Ori_Recording_11.wav        11         769.32        38              6   
11  Ori_Recording_12.wav        12        1799.35        8

**运行结果预览**：
- 每个Ori_Recording_XX.wav已被切割为60秒bag
- 已根据Burst/Buzz/Click的精确时间给bag打上1/0标签
- 统计表格已保存到 `results/phase0_bag_stats.csv`

Phase 0 完成，下一步可直接运行 Phase 1。